In [1]:
# Loading the google colab package and mounting the google drive to google colab
from google.colab import drive

drive.mount('/content/drive')

# Create the log directory
!mkdir '/content/drive/MyDrive/logs'

# Create the models directory
!mkdir '/content/drive/MyDrive/models'

Mounted at /content/drive
mkdir: cannot create directory ‘/content/drive/MyDrive/logs’: File exists
mkdir: cannot create directory ‘/content/drive/MyDrive/models’: File exists


In [2]:
!pip install tensorboardX ray

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 12.6 MB/s eta 0:00:00


In [3]:
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import tqdm.notebook as tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import normalize, to_categorical
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from keras.layers import BatchNormalization
from tensorflow.keras import optimizers
from tensorflow.keras.layers import Input, Dense, Activation, LSTM, GRU, SimpleRNN, Conv1D, TimeDistributed, MaxPooling1D, Flatten, Dropout

import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset # This will be used to turn the data into tensors
from torch.utils.data import DataLoader
from tensorboardX import SummaryWriter
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Import the hyperparameter tuning modules
from ray import tune
from ray.train import Checkpoint
from ray.air import session
from ray.tune.schedulers import ASHAScheduler
from functools import partial

In [4]:
# Setting the random seeds for reproductibility
np.random.seed(42)
random.seed(42)

# Importing the Data
delay_app = pd.read_csv('/content/drive/MyDrive/Data2/DelayAPP.csv')
gain_rpm = pd.read_csv('/content/drive/MyDrive/Data2/GainRPM.csv')
healthy_data = pd.read_csv('/content/drive/MyDrive/Data2/HealthyData.csv')
noise_app = pd.read_csv('/content/drive/MyDrive/Data2/NoiseAPP.csv')
packetLoss_app = pd.read_csv('/content/drive/MyDrive/Data2/PacketLossAPP.csv')

healthy_data['class'] = 'H'
delay_app['class'] = 'F1'
gain_rpm['class'] = 'F2'
noise_app['class'] = 'F3'
packetLoss_app['class'] = 'F4'

In [5]:
# Function to split the data
def label_data(df, size=0.3):
    num_classes = len(df['class'].unique())
    class_list = list(df['class'])
    for i in range(len(class_list)):
        cls = str(class_list[i])
        if cls == 'H':
            class_list[i] = 0
        elif cls == 'F1':
            class_list[i] =1
        elif cls == 'F2':
            class_list[i] = 2
        elif cls == 'F3':
            class_list[i] = 3
        elif cls == 'F4':
            class_list[i] = 4
        else:
            print(cls)
    df['class'] = class_list

    df['class'] = df['class'].astype(int)

    return df

In [6]:
def make_sequences(combined_data):
    dataframe = combined_data
    features = dataframe.columns.tolist()[1:8]
    X_train_data = dataframe[features]
    y_train_data = dataframe['class']

    X_train_data = normalize(X_train_data, axis=1)

    sc = StandardScaler()
    scaled_X_df = pd.DataFrame(sc.fit_transform(X_train_data))
    dataframe[features] = scaled_X_df
    df = dataframe
    cols = ['Timestamp']
    df[cols] = df[cols].applymap(np.int64)
    X_train = []
    y_train = []

    window_size = 20
    for b, group in df.groupby("Timestamp"):
      for d,e in group.groupby("class"):
        for i in range(len(e) - window_size + 1):
          temp = e[i: i + window_size]
          sequence_features = temp[features]
          X_train.append(sequence_features)
          y_train.append(d)


    y_train_cat = to_categorical(y_train)

    X_train = np.array(X_train)
    X_train = X_train.reshape(-1, 20, 7)
    y_train_arr = np.array(y_train_cat)

    X_train, X_test, y_train, y_test = train_test_split(X_train, y_train_arr, test_size=0.28, random_state=42)
    return X_train, X_test, y_train, y_test

## DELAY GRU

In [7]:
combined_data = pd.concat([healthy_data, delay_app], ignore_index=True)
combined_data = combined_data.dropna(axis=1)

df = label_data(combined_data)
X_train, X_test, y_train, y_test = make_sequences(combined_data)

In [8]:
y_train.shape

(606, 2)

In [9]:
number = 606
divisors = []

for i in range(1, number + 1):
    if number % i == 0:
        divisors.append(i)

print(divisors)


[1, 2, 3, 6, 101, 202, 303, 606]


In [42]:
class GRUNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes=2, num_layers=1):
        self.hidden_size = 100
        self.input_size = input_size
        self.num_layers=num_layers
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(self.input_size, self.hidden_size, self.num_layers, batch_first=True)
        self.fc = nn.Linear(self.hidden_size, 2)

    def forward(self, x):
        output, _ = self.gru(x)
        output = self.fc(output)
        output = output.mean(dim=0)
        print("Output Shape: {}".format(output.shape)) # Applying linear transformation
        probabilities = f.softmax(output, dim=1)  # Applying softmax across timesteps
        return probabilities


In [11]:
class SensorDataset(torch.utils.data.IterableDataset): # this class has some function to
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.data = zip([self.X, self.y])
        self.length = len(self.X)

    def __len__(self):
        return self.length
    # The logic for the multiple workers
    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            self.data = zip(*[self.X, self.y])
        else:
            worker_id = worker_info.id
            split_size = self.length//worker_info.num_workers
            start_index = worker_id * split_size
            end_index = (worker_id + 1) * split_size
            self.data = zip(*[self.X[start_index: end_index],
                        self.y[start_index: end_index]])

        return iter(self.data)

In [12]:
def normalize_dataset(X, y):
    X_means = []
    X_stds = []

    y_means = []
    y_stds = []

    # Normalize each column
    sensors = [column for column in X.columns if 'Sensor' in column]
    for sensor in sensors:
        # Get the values for the current sensor column
        X_values = X[sensor].values
        y_values = y[sensor].values

        # Perform normalization
        X_values_mean = X_values.mean()
        X_values_std = X_values.std()
        normalized_X_values = (X_values - X_values_mean) / X_values_std

        y_values_mean = y_values.mean()
        y_values_std = y_values.std()
        normalized_y_values = (y_values - y_values_mean) / y_values_std

        # Replace the original values with the normalized values
        X[sensor] = normalized_X_values
        y[sensor] = normalized_y_values

        # Save the means and standard deviations
        X_means.append(X_values_mean)
        X_stds.append(X_values_std)

        y_means.append(y_values_mean)
        y_stds.append(y_values_std)

    X_means = np.array(X_means)
    X_stds = np.array(X_stds)

    y_means = np.array(y_means)
    y_stds = np.array(y_stds)

    return X, X_means, X_stds, y, y_means, y_stds

In [13]:
def get_loaders(train_X, train_y, test_X, test_y, config, constant_params):

    # extract batch size, num_workers and pin memory from the hyperparameter list
    batch_size = config['batch_size']
    num_workers = constant_params['num_workers']
    pin_memory = constant_params['pin_memory']

    train_dataset = SensorDataset(train_X, train_y) # Create the training tensor dataset from pandas dataframes
    test_dataset = SensorDataset(test_X, test_y) # Create the test tensor dataset from pandas dataframes

    # Initialize PyTorch Data loaders for both training and test datasets

    train_loader = DataLoader(
        train_dataset,
        batch_size = batch_size,
        num_workers = num_workers,
        pin_memory = pin_memory,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size = batch_size,
        num_workers = num_workers,
        pin_memory = pin_memory,
        shuffle=False
    )

    return train_loader, test_loader

In [36]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


# Next is to write a function that trains the model for one training loop

def training_graph(config, constant_params):

    # Unpacking the config
    LEARNING_RATE = config['learning_rate']
    hidden_size = 100 #config['hidden_size']
    num_layers = config['num_layers']
    input_size = constant_params['input_size']
    seed = constant_params['seed']

    # Initialize the model
    model = GRUNet(input_size= input_size, hidden_size=100, num_layers=num_layers)

    # Specify the loss function

    loss_fn = nn.CrossEntropyLoss()

    # Initialize the optimizer and tensorboard writer

    optimizer = optim.AdamW(model.parameters(), lr= LEARNING_RATE)
    writer = SummaryWriter('/content/drive/MyDrive/logs')
    return model, loss_fn, optimizer, writer




def train_one_epoch(model, loss_fn, optimizer, writer, train_loader, learning_rate, epoch, config, constant_params):
    NUM_EPOCHS = config['num_epochs']
    DEVICE = constant_params['device']

    train_loss = 0.0
    train_steps = 0.0


    # train ONCE from start to end of data
    for data, labels in train_loader:
        data = data.float().to(device=DEVICE)
        labels = labels.float().to(device=DEVICE)
        model = model.to(DEVICE)
        output = model(data)
        print(labels.shape)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(output, labels) # VERY IMPORTANT! GRU DOES NOT HAVE LABEL DATA, BUT IT USES THE TRAINING DATA AS THE LABEL/TARGET
        loss.backward()
        train_loss += loss.item()
        train_steps += 1
        writer.add_scalar('loss', loss, epoch)
        for name, param in model.named_parameters():
          writer.add_histogram(name + '/grad', param.grad, epoch)
        for g in optimizer.param_groups:
          g['lr'] = learning_rate
        optimizer.step()


    output = {'loss': train_loss/train_steps

    }
    return output

In [15]:
def train_fn(model_class, config, constant_params, learning_rate_schedule = [(0.0, 1.0e-8), \
                                      (0.2, 0.02),    \
                                      (0.6, 0.01),   \
                                      (0.9, 1.0e-4), \
                                      (1.0, 1.0e-8) ]):
    NUM_EPOCHS = config['num_epochs']
    desc = constant_params['desc']
    lr = config['learning_rate']
    # Specifying the learning rate schedule for OneCycle
    lr_schedule_epochs, lr_schedule_rates = zip(*learning_rate_schedule)

    # Configuring the ray tune session to pick up where it left off

    checkpoint = session.get_checkpoint()

    if checkpoint:
      checkpoint_state = checkpoint.to_dict()
      start_epoch = checkpoint_state['epoch']
      model_class.model.load_state_dict(checkpoint_state['net_state_dict'])
      model_class.optimizer.load_state_dict(checkpoint_state['optimizer_state_dict'])
    else:
      start_epoch = 0

    for epoch in tqdm.tqdm(range(start_epoch, NUM_EPOCHS), desc=desc):
        learning_rate = lr   # np.interp(epoch / NUM_EPOCHS, lr_schedule_epochs, lr_schedule_rates)
        output = train_one_epoch(model_class.model,
                        model_class.loss_fn,
                        model_class.optimizer,
                        model_class.writer,
                        model_class.train_loader,
                        learning_rate,
                        epoch,
                        config,
                        constant_params)

        if epoch == (NUM_EPOCHS-1):
          print('Closing writer')
          model_class.writer.close()

        session.report({'reconstruction_loss': output['reconstruction_loss']})
    return output

In [16]:
# We need to create the model class to enable easy combination of all the different components
class ModelClass():
    def __init__(self, X_raw, y_raw, X_test, y_test):
        self.X_train = X_raw
        self.y_train = y_raw
        self.X_test = X_test
        self.y_test = y_test

    def build_graph(self, config, constant_params):
        self.model, \
        self.loss_fn, \
        self.optimizer, \
        self.writer = training_graph(config, constant_params)

    def prepare(self, config, constant_params):
        print("X_train's dimensions are {}".format(self.X_train.shape))
        print("y_train's dimensions are {}".format(self.y_train.shape))
        print("X_test's dimensions are {}".format(self.X_test.shape))
        print("y_test's dimensions are {}".format(self.y_test.shape))
        self.m, self.n, self.l = self.X_train.shape
        self.batch_size = config['batch_size'] # max(421, self.m // config[0])
        self.build_graph(config, constant_params)
        self.train_loader, self.test_loader = get_loaders(self.X_train, self.y_train, self.X_test, self.y_test, config, constant_params) # train_X, train_y, test_X, test_y

    def train(self, config, constant_params):
        output = train_fn(self, config=config, constant_params=constant_params)
        self.session_output = output

    def predict_values(self, constant_params):
        DEVICE = constant_params['device']

        X_test = torch.from_numpy(self.X_test.to_numpy()).float()
        y_test = torch.from_numpy(self.y_test.to_numpy()).float()

        X_test = X_test.to(DEVICE)
        y_test = y_test.to(DEVICE)

        self.model = self.model.to(DEVICE)

        predictions = self.model(X_test)

        val_loss = self.loss_fn(predictions, y_test).item()
        predictions =   predictions.cpu().detach().numpy()

        #predictions = self.X_test_means + self.X_test_stds * predictions # Reversing the normalization

        return predictions, val_loss

In [17]:
# Finally the function to run everything

def runModel(config, constant_params):
    # Unpacking the required constant parameters
    X_train = constant_params['training_data']
    y_train = constant_params['label_data']
    X_test = constant_params['X_test']
    y_test = constant_params['y_test']

    trained_model = constant_params['trained_model']

    # Creating the train and test datasets
    print("Splitting the data into training and test datasets")
    #X_train, y_train,X_test, y_test  = train_test_split(X, y, test_size=0.2, random_state=42) # Splits the data into 80% training and 20% test
    print("Done")

    # Initializing the Model Class
    print("Initializing Model Class")
    AI = ModelClass(X_train, y_train, X_test, y_test)
    print("Done")

    if trained_model is not None: # If you have passed an already trained model, take the trained model and continue training it
        AI.model = trained_model

        AI.prepare(config=config,constant_params=constant_params)
        t0 = time.time()
        AI.train(config=config, constant_params=constant_params)
        predictions, val_loss = AI.predict_values(constant_params)
        t1 = time.time()

        trained_model = AI.model

        y_test = AI.y_test_means + AI.y_test_stds * y_test
    else:
        AI.prepare(config=config,constant_params=constant_params)
        t0 = time.time()
        AI.train(config=config, constant_params=constant_params)
        predictions, val_loss = AI.predict_values(constant_params)
        t1 = time.time()

        trained_model = AI.model

        #y_test = AI.y_test_means + AI.y_test_stds * y_test

    output = {
        'y_test': y_test,
        'predictions': predictions,
        'val_loss': val_loss,
        'trained_model': trained_model,
        'loss': AI.session_output['loss']
    }

    return output


# Function to graph the results

def graph(title, predictions, test_data, xAxis, xAxisName, yAxisName, val_loss, computeMAD=False):
  numRows = test_data.shape[1]
  numCols = 1

  cols = list(test_data.columns)

  fig, ax = plt.subplots(numRows, numCols, squeeze=False)
  fig.set_size_inches(4 * numCols + 1.5, 4*numRows)

  for i, feature in enumerate(cols):
    if computeMAD:
        errors = 100 * np.abs(predictions - test_data)
        mad = np.mean(errors, axis=0)
        #print(rmse)
        t = "Validation Loss: {}, MAD: {}".format(val_loss, mad[i])
    else:
        t = "Validation Loss: %.2f" % val_loss

    ax[i, 0].annotate('%s' % feature, xy=(0, 0.5),
                      xytext=(-ax[i, 0].yaxis.labelpad-5, 0),
                      xycoords=ax[i, 0].yaxis.label, textcoords='offset points',
                      ha = 'right', va='center')

    ax[i, 0].set_xlabel(t)
    ax[i, 0].set_ylabel(yAxisName)

    ax[i, 0].plot(xAxis, predictions[:,i], 'co', markersize=2, markerfacecolor='white', label='predicted')

    ax[i, 0].plot(xAxis, test_data[feature], 'r.', markersize=0.5, label='targets')

    ax[i, 0].legend(prop={'size': 8}, loc='upper left')

  plt.tight_layout()
  plt.subplots_adjust(top=0.9)
  plt.suptitle("% s" % (title), fontsize=16)
  plt.show()

In [18]:
# Hyperparameter tuning
# TODO Ensure that the best model after hyperparameter tuning is the true best
def hyperparameterTuning(config, constant_params):
  # unpack the parameters
  max_num_epochs = constant_params['max_num_epochs']
  num_samples = constant_params['num_samples']

  # set the scheduler
  scheduler = ASHAScheduler(
      metric = 'loss',
      mode = 'min',
      max_t=max_num_epochs,
      grace_period=1,
      reduction_factor=2,
  )

  # run the hyperparameter search
  result = tune.run(
      partial(runModel, constant_params=constant_params),
      resources_per_trial={'cpu':2, "gpu": 1},
      num_samples = num_samples,
      config=config,
      scheduler=scheduler,
  )

  # Print the results
  best_trial = result.get_best_trial('loss','min','last')
  print(f"Best trial config: {best_trial.config}")
  print(f"Best trial final reconstruction loss: {best_trial.last_result['loss']}")
  best_checkpoint = result.get_best_checkpoint(best_trial, metric="loss", mode='min')
  return best_checkpoint

In [23]:
# Initializing some hyperparameters
# Performing initial training
LEARNING_RATE = 1e-8
BATCH_SIZE = 421
NUM_WORKERS = 2
PIN_MEMORY = True
DEVICE = 'cuda:0'
NUM_EPOCHS = 100
INPUT_SIZE = 7
HIDDEN_SIZE = 30
NUM_LAYERS = 4

In [20]:
seed = 5
set_seed(seed)

In [21]:
y_test.shape

(236, 2)

In [43]:
description = "Training GRU"

# TODO look for a way to save the best model during the hyperparameter tuning to then avoid having to run it again.
# Defining constant parameters
constant_params = {
    'num_workers' : NUM_WORKERS,
    'pin_memory' : PIN_MEMORY,
    'device' : DEVICE,
    'seed': seed,
    'desc': description,
    'training_data': X_train,
    'label_data': y_train,
    'X_test': X_test,
    'y_test': y_test,
    'trained_model': None,
    'input_size': INPUT_SIZE,
    'max_num_epochs': 150,
    'num_samples': 15,
    'model_type': 'GRU'
}

# Defining the parameter search space
config1 = {
    'batch_size' : tune.choice([421, 842]),
    'learning_rate' : tune.choice([1e-2,1e-3]),
    'num_epochs' : tune.choice([100]),
    'hidden_size': tune.choice([10, 20, 30,40,50]),
    'num_layers': tune.choice([2,4,6,8]),}

config = {
    'batch_size' : 20,
    'learning_rate' : 1e-2,
    'num_epochs' : 100,
    'hidden_size':25,
    'num_layers': 8,}
# Training using hyperparameter tuning

best_gru_1pct_noise = runModel(config, constant_params)

Splitting the data into training and test datasets
Done
Initializing Model Class
Done
X_train's dimensions are (606, 20, 7)
y_train's dimensions are (606, 2)
X_test's dimensions are (236, 20, 7)
y_test's dimensions are (236, 2)


Training GRU:   0%|          | 0/100 [00:00<?, ?it/s]

Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20, 2])
torch.Size([20, 2])
Output Shape: torch.Size([20

ValueError: ignored